# NIH14 Multilabel Manual Weight Editing

This notebook mirrors the manual weight editing experiment from the original Label-Free CBM paper for the NIH14 multilabel setup. Each pathology is a standalone binary classifier; edit a row of `W_c`, rebuild the CBM, and compare metrics for that class.


In [ ]:
import os
import json
from pathlib import Path

import torch
import numpy as np

os.chdir('..')

import cbm
import data_utils
from evaluate_cbm_nih import _load_thresholds


In [ ]:
MODEL_DIR = 'saved_models/nih14_example'
DEVICE = 'cuda'
CLASS_NAME = 'Pneumonia'
THRESHOLDS_PATH = 'saved_models/nih14_example/thresholds.json'
THRESHOLD_FALLBACK = 0.5


In [ ]:
model = cbm.load_cbm(MODEL_DIR, DEVICE)
model.eval()

with open(Path(MODEL_DIR) / 'args.txt') as f:
    args = json.load(f)
classes_path = data_utils.LABEL_FILES['nih14']
with open(classes_path) as f:
    classes = [c for c in f.read().splitlines() if c]
class_idx = classes.index(CLASS_NAME)
print(f'Loaded model covering {len(classes)} pathologies. Editing class {CLASS_NAME} (index {class_idx}).')

try:
    thresholds = _load_thresholds(THRESHOLDS_PATH, classes)
    threshold_vec = thresholds
except Exception as exc:
    threshold_vec = None
    print('Threshold file missing; falling back to scalar threshold when evaluating.', exc)


In [ ]:
W_c_path = Path(MODEL_DIR) / 'W_c.pt'
W_c = torch.load(W_c_path)
print('Weight matrix shape:', tuple(W_c.shape))
original_row = W_c[class_idx].clone()
print('First 20 concept weights for the class:')
for idx, weight in enumerate(original_row[:20]):
    print(f'{idx:03d} {weight.item():.4f}')


## Edit weights
Adjust `W_c[class_idx]` to remove or boost concepts. The example zeros the 10 most negative weights.

In [ ]:
row = W_c[class_idx].clone()
neg_idx = torch.topk(-row, k=10).indices
row[neg_idx] = 0.0
W_c[class_idx] = row

torch.save(W_c, Path(MODEL_DIR) / 'W_c_edited.pt')
print('Saved edited weights to W_c_edited.pt')


## Reload CBM with edited weights

In [ ]:
edited_model = cbm.load_cbm(MODEL_DIR, DEVICE)
edited_model.eval()
edited_model.W_c.data = torch.load(Path(MODEL_DIR) / 'W_c_edited.pt').to(DEVICE)


## Evaluate per-class metrics
Uses the same dataloader logic as `evaluate_cbm_nih.py` but restricted to `CLASS_NAME`.

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

split_name = 'nih14_val'
views = ['PA']

data_utils.configure_nih_dataset(img_dir=args['nih_img_dir'],
                                 csv_path=args.get('nih_csv_path'),
                                 train_fraction=args.get('nih_train_fraction', 0.9),
                                 split_seed=args.get('nih_split_seed', 0),
                                 views=views)
_, preprocess = data_utils.get_target_model(args['backbone'], 'cpu')
dataset = data_utils.get_data(split_name, preprocess)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

def collect_probs(active_model):
    active_model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for images, target in tqdm(loader):
            images = images.to(DEVICE)
            logits, _ = active_model(images)
            probs.append(torch.sigmoid(logits)[:, class_idx].cpu())
            labels.append(target[:, class_idx])
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

probs_before, labels = collect_probs(model)
probs_after, _ = collect_probs(edited_model)
threshold = threshold_vec[class_idx] if threshold_vec is not None else THRESHOLD_FALLBACK

def summarize(probs):
    auc = roc_auc_score(labels, probs)
    ap = average_precision_score(labels, probs)
    preds = (probs > threshold).astype(float)
    acc = (preds == labels.numpy()).mean()
    return {'auroc': auc, 'ap': ap, 'accuracy': acc}

print('Baseline:', summarize(probs_before))
print('Edited  :', summarize(probs_after))
